# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [66]:
#Load the data from aviation data to a pandas dataframe
df=pd.read_csv('aviation_data.csv', encoding='cp1252')
# Inspect NaNs and datatypes
print(df.info())

/tmp/ipykernel_2020/306980153.py:2: DtypeWarning: Columns (6: Latitude, 7: Longitude, 28: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('aviation_data.csv', encoding='cp1252')


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [67]:
#Filtering aircrafts older than 40 years (fron 1983 doing back)by using the Event.Date columnt.
df_filtered = df[df['Event.Date'] >= '1983-01-01']

#Filtering aircrafts that are amature built using the Amateur.Built column.
df_filtered = df_filtered[df_filtered['Amateur.Built'] == 'No']

#Filtering aircrafts from helocopters using the Aircraft.Category column.
df_filtered = df_filtered[df_filtered['Aircraft.Category'] == 'Airplane']

df_filtered.info()


<class 'pandas.DataFrame'>
Index: 21447 entries, 4149 to 88886
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                21447 non-null  str    
 1   Investigation.Type      21447 non-null  str    
 2   Accident.Number         21447 non-null  str    
 3   Event.Date              21447 non-null  str    
 4   Location                21441 non-null  str    
 5   Country                 21446 non-null  str    
 6   Latitude                19169 non-null  object 
 7   Longitude               19163 non-null  object 
 8   Airport.Code            13983 non-null  str    
 9   Airport.Name            14070 non-null  str    
 10  Injury.Severity         20634 non-null  str    
 11  Aircraft.damage         20220 non-null  str    
 12  Aircraft.Category       21447 non-null  str    
 13  Registration.Number     21241 non-null  str    
 14  Make                    21444 non-null  str    
 15

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [68]:
#filling the following columns we will use , Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
df_filtered['Total.Fatal.Injuries'].isna().sum()
df_filtered['Total.Fatal.Injuries'] = df_filtered['Total.Fatal.Injuries'].fillna(1)#mean is 0.74

df_filtered['Total.Serious.Injuries'].isna().sum()#total Nans is 2828
df_filtered['Total.Serious.Injuries'] = df_filtered['Total.Serious.Injuries'].fillna(1)#mean is 0.7408675188532919

df_filtered['Total.Minor.Injuries'].isna().sum()#total Nans is 2444
df_filtered['Total.Minor.Injuries'] = df_filtered['Total.Minor.Injuries'].fillna(0)#mean is 0.23059831772734488

df_filtered['Total.Uninjured'].isna().sum()#total Nans is 711
df_filtered['Total.Uninjured'] = df_filtered['Total.Uninjured'].fillna(8)#mean is 7.718

In [69]:
#estimate total number of passengers on board by summing up the total number of fatal,serious,minor and uninjured passengers
df_filtered['Total.Passengers'] = df_filtered['Total.Fatal.Injuries'] + df_filtered['Total.Serious.Injuries'] + df_filtered['Total.Minor.Injuries'] + df_filtered['Total.Uninjured']
#Estimate the likelihood of fatal or serious injuries by dividing the total number of fatal and serious injuries by the total number of passengers on board
df_filtered['Likelihood.Fatal.Serious'] = (df_filtered['Total.Fatal.Injuries'] + df_filtered['Total.Serious.Injuries']) / df_filtered['Total.Passengers']

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [70]:
#imputing the Aircraft.damage column with mode value because it is categorical column
df_filtered["Aircraft.damage"].isna().sum()#total Nans is 1227
df_filtered["Aircraft.damage"] = df_filtered["Aircraft.damage"].fillna(df_filtered["Aircraft.damage"].mode()[0])
#create new column is_destroyed to indicate if the aircraft was destroyed or not based on the Aircraft.damage column
#df_filtered['is_destroyed'] = df_filtered['Aircraft.damage'].apply(lambda x: 1 if x == 'Destroyed' else 0)
df_filtered['is_destroyed'] = df_filtered['Aircraft.damage'] == 'Destroyed'


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

##### ****CLEANING STEPS****
Convert the values in the Make column to lowercase to avoid duplicates due to case sensitivity</br>
</br>
Before stripping the count of unique values in the make column was 1332 after converting to lowercase and stripping the count of unique values in the make column is 1087</br>
</br>
Imputing the Nans(count was 3 only) with mode value because it is a categorical column and we cant drop the rows

In [71]:
df_filtered['Make'] = df_filtered['Make'].str.lower().str.strip()
df_filtered["Make"].isna().sum()#check for Nans, here nans =3

df_filtered["Make"] = df_filtered["Make"].fillna(df_filtered["Make"].mode()[0])#imputing the Nans with mode value because it is a categorical column
df_filtered["Make"].value_counts()#check for unique values in the Make column

Make
cessna                 7149
piper                  3989
beech                  1431
boeing                 1264
mooney                  363
                       ... 
mulholland robert a       1
detrick donald g          1
phantom                   1
james r dernovsek         1
orlican s r o             1
Name: count, Length: 1087, dtype: int64

In [72]:
#keep Makes with a reasonable number (you can put the threshold at 50
make_counts = df_filtered['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df_filtered= df_filtered[df_filtered['Make'].isin(valid_makes)]
df_filtered.info()


<class 'pandas.DataFrame'>
Index: 17895 entries, 4150 to 88886
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Event.Id                  17895 non-null  str    
 1   Investigation.Type        17895 non-null  str    
 2   Accident.Number           17895 non-null  str    
 3   Event.Date                17895 non-null  str    
 4   Location                  17891 non-null  str    
 5   Country                   17894 non-null  str    
 6   Latitude                  15990 non-null  object 
 7   Longitude                 15987 non-null  object 
 8   Airport.Code              11654 non-null  str    
 9   Airport.Name              11762 non-null  str    
 10  Injury.Severity           17176 non-null  str    
 11  Aircraft.damage           17895 non-null  str    
 12  Aircraft.Category         17895 non-null  str    
 13  Registration.Number       17729 non-null  str    
 14  Make               

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [73]:
#Get rid of nans in model , here we will drop the nans becouse the model acts as an identifier for the aircraft and it is important to have this information for our analysis
df_filtered["Model"].isna().sum()#check for Nans, here nans = 15
df_filtered = df_filtered.dropna(subset=["Model"])#drop the nans row in the dataframe 
df_filtered["Model"].isna().sum()#check for Nans, here nans = 0

np.int64(0)

In [74]:
#Inspect the column and counts for each model/make. Are model labels unique to each make?

model_make=df_filtered.groupby("Model")["Make"].nunique()
model_make[model_make > 1]
# models  appear under more than one make so we make a new column that combines the make and model to create a unique identifier for each aircraft

Model
100      2
112      2
112A     2
140      2
190      2
        ..
SR-20    2
SR-22    2
SR20     2
SR22     2
SR22T    2
Name: Make, Length: 116, dtype: int64

In [75]:
#create a new column that combines the make and model to create a unique identifier for each aircraft
df_filtered["Make_Model"] = df_filtered["Make"] + " " + df_filtered["Model"]
print("Distinct types:", df_filtered["Make_Model"].nunique())
# 6. Derived unique identifier per plane type
df_filtered["type_id"] = df_filtered.groupby("Make_Model").ngroup()

Distinct types: 2160


In [76]:
#  Checks
print(df_filtered["type_id"].nunique())   # should equal Make_Model.nunique()
type_lookup = (df_filtered[["type_id", "Make_Model"]]
               .drop_duplicates()
               .sort_values("type_id"))
print(type_lookup.tail(20))

2160
       type_id            Make_Model
80586     2140      taylorcraft BC12
75549     2141  taylorcraft BC12 - D
77540     2142   taylorcraft BC12 65
74284     2143    taylorcraft BC12 D
59058     2144   taylorcraft BC12-65
53815     2145    taylorcraft BC12-D
67318     2146   taylorcraft BC12-D1
24818     2147     taylorcraft BC12D
88569     2148  taylorcraft BC12D-85
67028     2149    taylorcraft BCS-65
66849     2150      taylorcraft BF12
77940     2151        taylorcraft BL
74223     2152     taylorcraft BL 65
70465     2153     taylorcraft BL-65
56791     2154     taylorcraft DC-65
63615     2155    taylorcraft DCO-65
55218     2156      taylorcraft F-19
59364     2157       taylorcraft F19
50982     2158      taylorcraft F21B
82698     2159      taylorcraft L 2M


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [77]:
#engine type cleaning
print(df_filtered['Engine.Type'].value_counts())#GET VALUE COUNTS FOR ENGINE TYPE COLUMN
print("null values is: " ,df_filtered['Engine.Type'].isna().sum())#check for Nans, here nans = 3214 ,there is already a value for unkowns so we will impute it with the "uknown"
df_filtered['Engine.Type'] = df_filtered['Engine.Type'].fillna("Unknown")#UNKNOWN IS NOW 3319

Engine.Type
Reciprocating      12835
Turbo Prop           931
Turbo Fan            701
Unknown              106
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64
null values is:  3214


In [78]:
#CLEANINING WEATHER CONDITION COLUMN
print(df_filtered['Weather.Condition'].value_counts())#GET VALUE COUNTS FOR WEATHER CONDITION COLUMN
print("null values is: " ,df_filtered['Weather.Condition'].isna().sum())
#CLEAN THE VALUES BY STRIPING AND LOWERING THE CASE
df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].str.lower().str.strip()
#FILL UNKNOWN VALUES WITH "unknown"
df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].replace(['Unk', 'UNK'], 'Unknown').fillna("unk")

Weather.Condition
VMC    14295
IMC      905
Unk      186
UNK       77
Name: count, dtype: int64
null values is:  2417


In [79]:
#no.of engines cleaning
print(df_filtered['Number.of.Engines'].value_counts())#GET VALUE COUNTS FOR number of engines column
#drop the no of engine =0
df_filtered = df_filtered[df_filtered['Number.of.Engines'] != 0]#Dealing with extreme values,they were only 5 ,and an aicracft can't run with 0 engines
#get the null values for the number of engines column
print("null values is: " ,df_filtered['Number.of.Engines'].isna().sum())
df_filtered['Number.of.Engines'].mean()#mean is 1.1725 so we will impute it with 1 because we cant have a fraction of an engine and the mean is very close to 1 so we will impute it with 1
df_filtered['Number.of.Engines'] = df_filtered['Number.of.Engines'].fillna(1)

Number.of.Engines
1.0    13222
2.0     2470
4.0       67
3.0       26
0.0        5
Name: count, dtype: int64
null values is:  2090


In [80]:
#cleaning the purpose  of the flight column
df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].str.lower().str.strip()
df_filtered['Purpose.of.flight'].value_counts()
#fill in the nans with unknown
df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].fillna("unknown")
print(df_filtered['Purpose.of.flight'].value_counts())
#Deal with air race show where the air race /show will be replaced with air race show 
df_filtered= df_filtered.replace({'Purpose.of.flight': {'air race/show': 'air race show'}})

Purpose.of.flight
personal                     9843
unknown                      3349
instructional                2409
aerial application            724
business                      409
positioning                   269
skydiving                     157
aerial observation            147
other work use                121
banner tow                     86
flight test                    73
ferry                          72
executive/corporate            65
glider tow                     29
public aircraft - federal      28
public aircraft                26
public aircraft - state        21
air race show                  15
firefighting                   12
public aircraft - local        11
pubs                            3
air race/show                   2
air drop                        2
asho                            2
Name: count, dtype: int64


In [81]:
#cleaning broad.phase.flight column
df_filtered['Broad.phase.of.flight'] = df_filtered['Broad.phase.of.flight'].str.lower().str.strip()
print(df_filtered['Broad.phase.of.flight'].value_counts())
#fill in null values with unknown
df_filtered['Broad.phase.of.flight'] = df_filtered['Broad.phase.of.flight'].fillna("unknown")

Broad.phase.of.flight
landing        1109
takeoff         425
cruise          238
approach        210
maneuvering     127
taxi             99
go-around        81
descent          62
climb            52
standing         35
unknown          11
other             2
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
#removing columns with too many null values, we will drop the following columns because they have too many null values and they are not important for our analysis
df_filtered.isna().sum()#schedule has 15739 null values,air carrier has 9426 nulls so we drop them 
df_filtered = df_filtered.drop(['Schedule', 'Air.carrier'], axis=1)
print(df_filtered.isna().sum())


Event.Id                       0
Investigation.Type             0
Accident.Number                0
Event.Date                     0
Location                       4
Country                        1
Latitude                    1894
Longitude                   1897
Airport.Code                6227
Airport.Name                6121
Injury.Severity              717
Aircraft.damage                0
Aircraft.Category              0
Registration.Number          165
Make                           0
Model                          0
Amateur.Built                  0
Number.of.Engines              0
Engine.Type                    0
FAR.Description              344
Purpose.of.flight              0
Total.Fatal.Injuries           0
Total.Serious.Injuries         0
Total.Minor.Injuries           0
Total.Uninjured                0
Weather.Condition              0
Broad.phase.of.flight          0
Report.Status               3785
Publication.Date             788
Total.Passengers               0
Likelihood

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [86]:
#saving dataframe to a new csv file called cleaned_aviation_data.csv
df_filtered.to_csv('cleaned_aviation_data.csv', index=False)